In [ ]:
import ipywidgets as widgets
import math
import matplotlib.pyplot as plt
import numpy as np
import os
from ipyfilechooser import FileChooser
from scipy.interpolate import interp1d

# Calculates an approximation of Chandrasekhar's H-function, according to Hapke (2002, 2005).
# The H-function is used here to model multiply-scattered light.
# Inputs: ssa - single-scattering albedo value
#         cos - cosine of an angle
def h_function(ssa, cos):
    gamma = math.sqrt(1 - ssa)
    r0 = (1 - gamma) / (1 + gamma)
    h = 1 / (1 - (1 - gamma) * cos * (r0 + (1 - r0 / 2 - r0 * cos) * math.log((1 + cos) / cos)))
    return h

# Calculates the reflectance, providing a relationship with the single-scattering albedo (Hapke, 1981).
# This implementation uses the bidirectional radiance coefficient.
# Inputs: ssa - single-scattering albedo value
#         cosi - cosine of the incidence angle
#         cose - cosine of the emergence angle
#         pha - phase angle
#         phf - single particle phase function
#         opp - opposition effect
def r_function(ssa, cosi, cose, pha, phf, opp):
    h_cosi = h_function(ssa, cosi)
    h_cose = h_function(ssa, cose)
    ref = (1 / (4 * (cose + cosi))) * (ssa * (1 + opp) * phf + ssa * h_cose * h_cosi - ssa)
    return ref

# Calculates the single-scattering albedo at a given wavelength, given the optical constants and grain size (Hapke, 1993).
# Inputs: n - real index of refraction
#         k - imaginary index of refraction
#         d - grain size (in microns)
#         wav - wavelength (in microns)
def nkdwav2ssa(n, k, d, wav):
    alpha = 4 * math.pi * k / wav
    s = 0
    ri = (1 - math.sqrt(alpha / (alpha + s))) / (1 + math.sqrt(alpha / (alpha + s)))
    da = 2 / 3 * (n ** 2 - 1 / n * (n ** 2 - 1) ** (3 / 2)) * d
    theta = (ri + math.exp(-math.sqrt(alpha * (alpha + s)) * da)) / (1 + ri * math.exp(-math.sqrt(alpha * (alpha + s)) * da))
    se = ((n - 1) ** 2 + k ** 2) / ((n + 1) ** 2 + k ** 2) + 0.05
    si = 1.014 - 4 / (n * (n + 1) ** 2)
    ssa = se + (1 - se) * (1 - si) / (1 - si * theta) * theta
    return ssa

# Convert a reflectance/single-scattering albedo spectrum to an optical constants (n/k) spectrum.
# Plots the resulting spectra, and provides an option to save the results for usage in other applications.
def run(button):
    # Handle file saving. Needs to be an inner function to access the data variables.
    def save(button):
        output_path = output_fc.selected_path

        if output_path is None:
            with output:
                print("No output folder selected.")
            return

        # Save the generated SSA spectrum if the starting point was a reflectance spectrum.
        if input_type.value == "Reflectance":
            ssa_filename = os.path.splitext(input_fc.selected_filename)[0]
            ssa_ext = os.path.splitext(input_fc.selected_filename)[1]
            ssa_filepath = os.path.join(output_path, ssa_filename + "_ssa" + ssa_ext)
            np.savetxt(ssa_filepath, ssa_data, "%.6f")
            with output:
                print("Saved SSA spectrum to:", ssa_filepath)

        # Save the generated n/k spectrum.
        nk_filename = os.path.splitext(input_fc.selected_filename)[0]
        nk_ext = os.path.splitext(input_fc.selected_filename)[1]
        nk_filepath = os.path.join(output_path, nk_filename + "_nk" + nk_ext)
        np.savetxt(nk_filepath, nk_data, ["%.6f", "%.6f", "%.6e"])
        with output:
            print("Saved nk spectrum to:", nk_filepath)

    # Reset view every time the submit button is clicked.
    output.clear_output()

    # If the starting point is a reflectance spectrum, convert to SSA first before converting to n/k.
    # Otherwise, skip to the n/k conversion.
    input_path = input_fc.selected
    if input_type.value == "Reflectance":
        ref_data = np.loadtxt(input_path)
        cosi = math.cos(inc.value * math.pi / 180)
        cose = math.cos(eme.value * math.pi / 180)
        pha = inc.value + eme.value

        # Construct lookup table for conversion between SSA and reflectance.
        # The step size may be increased/decreased for coarser/finer resolution.
        step = 0.0001
        ssa = np.linspace(0.0, 1.0, int(1 / step))
        num_ssa = len(ssa)
        ssa_ref = np.zeros((num_ssa, 2))
        for i in range(num_ssa):
            ref = r_function(ssa[i], cosi, cose, pha, phf.value, opp.value)
            ssa_ref[i] = [ssa[i], ref]

        # Show that r_function is a nonlinear function of SSA and is strictly monotonic (i.e., the lookup table approach is valid).
        '''
        with output:
            plt.plot(ssa_ref[:, 0], ssa_ref[:, 1])
            plt.xlabel("Single-scattering albedo")
            plt.ylabel("Reflectance")
        '''

        # Convert reflectance spectrum to SSA.
        num_ref = len(ref_data)
        ssa_data = np.zeros((num_ref, 2))
        ssa_data[:, 0] = ref_data[:, 0]
        for i in range(num_ref):
            diff = abs(ssa_ref[:, 1] - ref_data[i, 1])
            min_index = np.argmin(diff)
            ssa_data[i, 1] = ssa_ref[min_index, 0]

        # Display reflectance and SSA spectra.
        with output:
            fig, (p1, p2) = plt.subplots(1, 2)
            p1.plot(ref_data[:, 0], ref_data[:, 1])
            p1.set_xlabel("Wavelength")
            p1.set_ylabel("Reflectance")
            p2.plot(ssa_data[:, 0], ssa_data[:, 1])
            p2.set_xlabel("Wavelength")
            p2.set_ylabel("Single-scattering albedo")
            fig.tight_layout()
            plt.show()
    elif input_type.value == "Single-scattering albedo":
        # Display SSA spectrum.
        ssa_data = np.loadtxt(input_path)
        with output:
            plt.plot(ssa_data[:, 0], ssa_data[:, 1])
            plt.xlabel("Wavelength")
            plt.ylabel("Single-scattering albedo")
            plt.show()

    # Convert between optical constants and SSA. For now, assume that n is constant, and we're only solving for k.
    # Since the upper bound of k is unknown, calculate on the fly instead of keeping a potentially large lookup table.
    # The step size may be increased/decreased for coarser/finer resolution.
    step = 1e-7
    num_ssa = len(ssa_data)
    nk_data = np.zeros((num_ssa, 3))
    nk_data[:, 0] = ssa_data[:, 0]
    nk_data[:, 1] = n.value
    for i in range(num_ssa):
        wav = nk_data[i, 0]
        n_i = nk_data[i, 1]
        ssa_i = ssa_data[i, 1]
        ssa_curr = ssa_prev = 0
        k_curr = k_prev = 0
        while True:
            if k_curr > 0: ssa_prev = ssa_curr
            ssa_curr = nkdwav2ssa(n_i, k_curr, d.value, wav)
            if (ssa_prev <= ssa_i <= ssa_curr) or (ssa_curr <= ssa_i <= ssa_prev): break
            k_prev = k_curr
            k_curr += step
        linear_interp = interp1d([ssa_prev, ssa_curr], [k_prev, k_curr]) # Best replicates MATLAB's interp1 function.
        nk_data[i, 2] = linear_interp(ssa_i)

    # Display n/k spectrum.
    with output:
        plt.plot(nk_data[:, 0], nk_data[:, 2])
        plt.xlabel("Wavelength")
        plt.ylabel("k")
        plt.show()

        # Make saving options available.
        save_button = widgets.Button(description = "Save spectra")
        save_button.on_click(save)
        output_fc.default_path = input_fc.selected_path
        display(output_fc)
        display(save_button)

# Setup user-input fields.
style = {"description_width": "initial"}
input_fc = FileChooser("")
input_fc.title = "Choose a spectrum:"
input_type = widgets.RadioButtons(options = ["Reflectance", "Single-scattering albedo"], description = "Type of input spectrum:", disabled = False)
inc = widgets.FloatText(value = 30, description = "Incidence angle (degrees):", style = style, disabled = False)
eme = widgets.FloatText(value = 0, description = "Emergence angle (degrees):", style = style, disabled = False)
phf = widgets.FloatText(value = 1, description = "Phase function:", style = style, disabled = False)
opp = widgets.FloatText(value = 0, description = "Opposition effect:", style = style, disabled = False)
n = widgets.FloatText(value = 3.05, description = "Real index of refraction:", style = style, disabled = False)
d = widgets.FloatText(value = 250, description = "Grain size (microns):", style = style, disabled = False)
convert_button = widgets.Button(description = "Convert")
convert_button.on_click(run)
output = widgets.Output()
output_fc = FileChooser("")
output_fc.title = "Save spectra to folder:"
output_fc.show_only_dirs = True
widgets.VBox([input_fc, input_type, inc, eme, phf, opp, n, d, convert_button, output])
